# 03 - Risk-aware decision engine

The engine encodes a metric hierarchy modeled on Microsoft ExP / Spotify practice:

| role | metric | test | multiplicity |
|---|---|---|---|
| primary success | checkout conversion | one-sided superiority z-test | BH-corrected |
| secondary success | revenue per user | one-sided superiority z-test | BH-corrected |
| guardrail | average order value | non-inferiority vs -2% margin | uncorrected, power reported |
| guardrail | checkout error rate | non-inferiority vs +0.5pp margin | uncorrected, power reported |

**Why one-sided superiority?** We ship only on improvement; "no effect" and "harm" both mean do-not-ship, so they share the null.

**Why is BH applied to success metrics only?** For success metrics the costly error is a FALSE claim of improvement (Type I), which multiplies with the number of metrics, so the family is FDR-corrected. For guardrails the costly error is MISSING real harm (Type II): alpha-correcting them would make harm even easier to miss. Instead each guardrail reports its power to clear the margin, and low power triggers INVESTIGATE rather than a silent pass.

**Verdicts**: SHIP (primary significant after BH + every guardrail non-inferior with adequate power), DO NOT SHIP (primary fails or harm beyond a margin is established), INVESTIGATE (a guardrail is inconclusive).

Note on the guardrails: a statistically significant degradation SMALLER than the margin does not block shipping. The margin is the business's declared tolerance; with big samples even accepted, trivial costs become statistically significant.

In [1]:
import sys
sys.path.insert(0, "..")  # so config, src and data import from the repo root

import config
from data.generate_data import generate_experiment_data
from src.power import required_sample_size_per_arm
from src.decision_engine import default_metric_specs, evaluate_experiment

n_default = required_sample_size_per_arm(
    config.CONTROL_CONVERSION_RATE, config.MDE_RELATIVE
)

## 1. The experiment as designed in notebook 01: verdict is INVESTIGATE

We first run the engine on the experiment at the sample size that powers the PRIMARY metric (70k per arm). The result is a finding in itself: the sample size that suffices for conversion leaves the average order value guardrail with only ~20% power, because AOV lives on the ~3.5% converter subpopulation and order values are heavy-tailed. Relative to what the guardrails need, the primary-powered sample size IS the underpowered case.

In [2]:
df = generate_experiment_data(n_users=2 * n_default, seed=204)
print(evaluate_experiment(df, default_metric_specs()).render())

# Decision report

## VERDICT: INVESTIGATE

- primary improved, but at least one guardrail is inconclusive

alpha = 0.05, guardrail power floor = 80%

### checkout conversion rate (success, proportion test)
- effect: +0.00319 (one-sided 95% lower bound +0.00156)
- one-sided superiority p = 0.0007; significant after BH: YES
- status: PASS

### revenue per user (success, mean test)
- effect: +0.18236 (one-sided 95% lower bound +0.05390)
- one-sided superiority p = 0.0098; significant after BH: YES
- status: PASS

### average order value (guardrail, mean test)
- effect: -0.15327, margin: -1.17951 (relative 0.02), worst-case bound: -2.52879
- non-inferiority p = 0.2387 (NOT established); power to clear margin if harmless = 20%
- status: INCONCLUSIVE  (non-inferiority not established: underpowered (20% < 80%))

### checkout error rate (guardrail, proportion test)
- effect: +0.00140, margin: +0.00500 (absolute 0.005), worst-case bound: +0.00237
- non-inferiority p = 0.0000 (established); pow

## 2. SHIP scenario

To power the AOV guardrail to ~90% against its 2% margin we need roughly 900k users per arm (heavy-tailed order values, converters-only population). We simulate a variant with no true AOV change at that traffic level: primary improves, both guardrails clear their margins with adequate power. Note the error-rate guardrail: the true +0.1pp degradation is statistically detected, but it is WITHIN the +0.5pp margin the business accepted, so it does not block the ship.

In [3]:
df_ship = generate_experiment_data(n_users=1_800_000, seed=202, aov_effect=0.0)
print(evaluate_experiment(df_ship, default_metric_specs()).render())

# Decision report

## VERDICT: SHIP

- primary metric 'checkout conversion rate' significantly improved after BH correction
- all guardrails non-inferior with adequate power

alpha = 0.05, guardrail power floor = 80%

### checkout conversion rate (success, proportion test)
- effect: +0.00292 (one-sided 95% lower bound +0.00246)
- one-sided superiority p = 0.0000; significant after BH: YES
- status: PASS

### revenue per user (success, mean test)
- effect: +0.18143 (one-sided 95% lower bound +0.14561)
- one-sided superiority p = 0.0000; significant after BH: YES
- status: PASS

### average order value (guardrail, mean test)
- effect: +0.26458, margin: -1.17519 (relative 0.02), worst-case bound: -0.38466
- non-inferiority p = 0.0001 (established); power to clear margin if harmless = 91%
- status: PASS

### checkout error rate (guardrail, proportion test)
- effect: +0.00101, margin: +0.00500 (absolute 0.005), worst-case bound: +0.00129
- non-inferiority p = 0.0000 (established); power to 

## 3. DO NOT SHIP scenario

Same traffic, but the true average order value effect is cranked to -5%, far past the -2% margin. The engine now establishes harm beyond the margin (the entire CI sits outside it) and blocks the ship even though the primary metric improved: exactly the trade-off guardrails exist to catch (the streamlined checkout converts more users but they buy noticeably less per order).

In [4]:
df_harm = generate_experiment_data(n_users=1_800_000, seed=203, aov_effect=-0.05)
print(evaluate_experiment(df_harm, default_metric_specs()).render())

# Decision report

## VERDICT: DO NOT SHIP

- harm beyond a guardrail margin is statistically established

alpha = 0.05, guardrail power floor = 80%

### checkout conversion rate (success, proportion test)
- effect: +0.00247 (one-sided 95% lower bound +0.00201)
- one-sided superiority p = 0.0000; significant after BH: YES
- status: PASS

### revenue per user (success, mean test)
- effect: +0.05098 (one-sided 95% lower bound +0.01577)
- one-sided superiority p = 0.0086; significant after BH: YES
- status: PASS

### average order value (guardrail, mean test)
- effect: -2.49843, margin: -1.17636 (relative 0.02), worst-case bound: -3.13583
- non-inferiority p = 0.9997 (NOT established); power to clear margin if harmless = 92%
- status: HARM  (harm beyond the margin is statistically established)

### checkout error rate (guardrail, proportion test)
- effect: +0.00077, margin: +0.00500 (absolute 0.005), worst-case bound: +0.00105
- non-inferiority p = 0.0000 (established); power to clear mar

## Takeaways

- The three verdicts respond to evidence the way a risk-aware review should: improvement alone is not enough; absence of evidence of harm is not evidence of absence (INVESTIGATE), and established harm vetoes the ship (DO NOT SHIP).
- Guardrails on converter-level mean metrics (like AOV) need far more traffic than the primary conversion test. In practice teams respond by widening the margin, switching to a more sensitive guardrail formulation, or accepting the Type II risk explicitly.
- The error-rate guardrail illustrates why DO-NOT-SHIP triggers on harm BEYOND THE MARGIN, not on any statistically significant harm: at large n, a +0.1pp accepted cost is highly significant but well within tolerance.
- `validate_phase3.py` verifies the machinery: NI test passes ~5% at the margin and ~power at zero harm (G3a), BH controls FDR (G3b), and the three scenarios yield their verdicts (G3c).